In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "bronze-rp")
dbutils.widgets.text("catalogo", "catalog_RP")
dbutils.widgets.text("esquema", "bronze")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

ruta = f"abfss://{container}@adlssmartdata1201rp.dfs.core.windows.net/Ecommerce_Products.csv"

In [0]:
Ecommerce_schema = StructType(
    fields=[
        StructField("Date", StringType(), False),
        StructField("Product_Category", StringType(), True),
        StructField("Price", DoubleType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Customer_Segment", StringType(), True),
        StructField("Marketing_Spend", DoubleType(), True),
        StructField("Units_Sold", IntegerType(), True),
    ]
)

In [0]:
df_Ecommerce_final = spark.read\
.option('header', True)\
.schema(Ecommerce_schema)\
.csv(ruta)

In [0]:
Ecommerce_selected_df = df_Ecommerce_final.select(
    col("Date"), col("Product_Category"), col("Price"), col("Discount"),
    col("Customer_Segment"), col("Marketing_Spend"), col("Units_Sold")
)

In [0]:
Ecommerce_renamed_df = (
    Ecommerce_selected_df.withColumnRenamed("Date", "Fecha")
    .withColumnRenamed("Product_Category", "CategoriaProducto")
    .withColumnRenamed("Price", "Precio")
    .withColumnRenamed("Discount", "Descuento")
    .withColumnRenamed("Customer_Segment", "SegmentoCliente")
    .withColumnRenamed("Marketing_Spend", "MarketingGastos")
    .withColumnRenamed("Units_Sold", "UndVedidas")
)

In [0]:
Ecommerce_final_df = Ecommerce_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
Ecommerce_final_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.Ecommerce")